In [ ]:
def train_model(graph, config, model, device='cuda'):
    """
    FIXED TRAINING: GNN trains via proxy gradient trick.
    - Edge MLP: trains every accumulation step (fast)
    - GNN encoder: trains once per epoch via accumulated gradients
    """
    physical_batch = config["physical_batch_size"]
    accum_steps = config["accumulation_steps"]
    type_w = config.get("type_loss_weight", 0.5)
    validate_every = config.get("validate_every_n_epochs", 2)

    START_TIME = time.perf_counter()
    print(f"MODEL TRAINING STARTING AT: {time.strftime('%H:%M:%S', time.localtime())}...\n")

    # Data Loaders
    train_edges = graph['edge_index'][:, graph['train_mask']]
    train_labels = torch.stack([
        graph['y_binary'][graph['train_mask']],
        graph['y_type'][graph['train_mask']]
    ], dim=1)
    val_edges = graph['edge_index'][:, graph['val_mask']]
    val_labels = torch.stack([
        graph['y_binary'][graph['val_mask']],
        graph['y_type'][graph['val_mask']]
    ], dim=1)

    train_loader = DataLoader(
        TensorDataset(train_edges.t(), train_labels),
        batch_size=physical_batch, shuffle=True,
        pin_memory=True, num_workers=2, persistent_workers=True
    )
    val_loader = DataLoader(
        TensorDataset(val_edges.t(), val_labels),
        batch_size=physical_batch * 2, shuffle=False,
        pin_memory=True, num_workers=2
    )

    # Class weights
    y_train_bin = graph['y_binary'][graph['train_mask']].numpy()
    n_pos, n_neg = int((y_train_bin == 1).sum()), int((y_train_bin == 0).sum())
    if config.get("use_class_weights", True) and n_pos > 0 and n_neg > 0:
        w_pos = (n_pos + n_neg) / (2.0 * n_pos)
        w_neg = (n_pos + n_neg) / (2.0 * n_neg)
        class_weights = torch.tensor([w_neg, w_pos], dtype=torch.float32, device=device)
        crit_bin = nn.CrossEntropyLoss(weight=class_weights)
        print(f"   Class weights: neg={w_neg:.3f}, pos={w_pos:.3f}")
    else:
        crit_bin = nn.CrossEntropyLoss()
    crit_type = nn.CrossEntropyLoss(ignore_index=-1)

    # Move graph to GPU
    full_adj = graph['edge_index'].to(device)
    x_v1 = graph['x_v1'].to(device)
    x_v2 = graph['x_v2'].to(device)
    x_v3 = graph['x_v3'].to(device)

    # ============================================================
    # KEY FIX: TWO SEPARATE OPTIMIZERS
    # GNN encoder params vs Edge classifier params
    # ============================================================
    gnn_param_names = ['proj_v1', 'proj_v2', 'proj_v3', 'feat_attention', 
                   'gat1', 'gat2', 'gat3', 'norm1', 'norm2', 'norm3']
    edge_param_names = ['edge_encoder', 'head_bin', 'head_type']

    # Get underlying module if torch.compile was used
    base_model = model._orig_mod if hasattr(model, '_orig_mod') else model

    gnn_params = [p for n, p in base_model.named_parameters()
                  if any(n.startswith(x) for x in gnn_param_names)]
    edge_params = [p for n, p in base_model.named_parameters()
                   if any(n.startswith(x) for x in edge_param_names)]

    print(f"   GNN params:  {sum(p.numel() for p in gnn_params):,}")
    print(f"   Edge params: {sum(p.numel() for p in edge_params):,}")

    gnn_optimizer  = Adam(gnn_params,  lr=config['lr'], weight_decay=config.get('weight_decay', 1e-4))
    edge_optimizer = Adam(edge_params, lr=config['lr'], weight_decay=config.get('weight_decay', 1e-4))

    # One scaler handles both — we call backward once for each
    edge_scaler = GradScaler()

    gnn_scheduler = lr_scheduler.ReduceLROnPlateau(
        gnn_optimizer, mode='max',
        patience=config.get('scheduler_patience', 4),
        factor=config.get('scheduler_factor', 0.5)
    )
    edge_scheduler = lr_scheduler.ReduceLROnPlateau(
        edge_optimizer, mode='max',
        patience=config.get('scheduler_patience', 4),
        factor=config.get('scheduler_factor', 0.5)
    )

    history = {
        'train_loss': [], 'val_acc': [], 'val_f1': [],
        'val_precision': [], 'val_recall': [], 'training_duration': []
    }

    print(f"\n{'—'*75}")
    print(f"{'Epoch':<6} | {'Train Loss':<11} | {'Val Acc':<8} | {'Val F1':<8} | {'Val P':<8} | {'Val R':<8} | {'Time':<8}")
    print(f"{'—'*75}")

    best_f1 = 0.0
    # ADD THIS LINE before the epoch loop:
    tracker = BestModelTracker(
        save_dir='models',
        min_delta=0.005,     # Only save if F1 improves by >0.5%
        metric_name='f1'
    )

    for epoch in range(1, config['epochs'] + 1):
        epoch_start = time.time()
        model.train()

        # ============================================================
        # PHASE 1: Compute GNN node embeddings WITH gradient tracking
        # This is the KEY fix — no torch.no_grad() here!
        # ============================================================
        gnn_optimizer.zero_grad()
        with autocast(device.type):
            node_emb = model.get_node_embeddings(x_v1, x_v2, x_v3, full_adj)

        # Proxy: detach for memory-efficient batch training,
        # but keep requires_grad=True so we can accumulate edge→node gradients
        node_emb_proxy = node_emb.detach().requires_grad_(True)

        # ============================================================
        # PHASE 2: Train Edge MLP using proxy embeddings (fast batches)
        # ============================================================
        edge_optimizer.zero_grad()
        total_loss = 0.0

        

        for i, (batch_edges, batch_labels) in enumerate(train_loader):
            batch_edges = batch_edges.t().to(device, non_blocking=True)
            y_bin  = batch_labels[:, 0].to(device, non_blocking=True)
            y_type = batch_labels[:, 1].to(device, non_blocking=True)

            with autocast(device.type):
                pred_bin, pred_type = model.forward_edges_from_emb(node_emb_proxy, batch_edges)
                loss = (crit_bin(pred_bin, y_bin) +
                        type_w * crit_type(pred_type, y_type)) / accum_steps

            edge_scaler.scale(loss).backward()   # Accumulates in node_emb_proxy.grad too
            total_loss += loss.item() * accum_steps

            if (i + 1) % accum_steps == 0:
                edge_scaler.step(edge_optimizer)
                edge_scaler.update()
                edge_optimizer.zero_grad()
                # Don't zero node_emb_proxy.grad — we keep accumulating!

        train_loss = total_loss / len(train_loader)
        history['train_loss'].append(train_loss)

        # ============================================================
        # PHASE 3: Propagate accumulated gradients back through GNN
        # This is what was MISSING before — GNN now actually learns!
        # ============================================================
        if node_emb_proxy.grad is not None:
            # Unscale edge_scaler's scale factor before propagating
            # node_emb_proxy.grad was accumulated under edge_scaler's AMP context,
            # so divide by edge_scaler's scale to get true gradient magnitude
            scale = edge_scaler.get_scale()
            unscaled_grad = node_emb_proxy.grad / scale if scale != 0 else node_emb_proxy.grad
            node_emb.backward(unscaled_grad)
            torch.nn.utils.clip_grad_norm_(gnn_params, max_norm=1.0)
            gnn_optimizer.step()             # ← Direct step, no scaler wrapping needed
        gnn_optimizer.zero_grad()

        # ============================================================
        # Validation
        # ============================================================
        do_validate = (epoch % validate_every == 0) or (epoch == config['epochs'])

        if do_validate:
            model.eval()
            all_preds, all_trues = [], []

            with torch.no_grad():
                with autocast(device.type):
                    node_emb_val = model.get_node_embeddings(x_v1, x_v2, x_v3, full_adj)
                for batch_edges, batch_labels in val_loader:
                    batch_edges = batch_edges.t().to(device, non_blocking=True)
                    p_bin, _ = model.forward_edges_from_emb(node_emb_val, batch_edges)
                    all_preds.extend(torch.argmax(p_bin, dim=1).cpu().numpy())
                    all_trues.extend(batch_labels[:, 0].numpy())

            all_trues = np.array(all_trues)
            all_preds = np.array(all_preds)

            val_acc  = accuracy_score(all_trues, all_preds)
            val_f1   = f1_score(all_trues, all_preds, zero_division=0)
            val_prec = precision_score(all_trues, all_preds, zero_division=0)
            val_rec  = recall_score(all_trues, all_preds, zero_division=0)

            history['val_acc'].append(val_acc)
            history['val_f1'].append(val_f1)
            history['val_precision'].append(val_prec)
            history['val_recall'].append(val_rec)

            gnn_scheduler.step(val_f1)
            edge_scheduler.step(val_f1)

            tracker.update(model, val_f1, epoch)
            if val_f1 > best_f1:
                best_f1 = val_f1

            epoch_time = time.time() - epoch_start
            print(f"{epoch:>4}   | {train_loss:>9.4f}   | {val_acc:>6.4f}  | {val_f1:>6.4f}  | {val_prec:>6.4f}  | {val_rec:>6.4f}  | {epoch_time:>5.1f}s")
        else:
            for k in ['val_acc', 'val_f1', 'val_precision', 'val_recall']:
                history[k].append(history[k][-1] if history[k] else 0.0)
            epoch_time = time.time() - epoch_start
            print(f"{epoch:>4}   | {train_loss:>9.4f}   | (no val) | {epoch_time:>5.1f}s")

        torch.cuda.empty_cache()
    
    tracker.summary()
    model = tracker.load_best(model)

    END_TIME = time.perf_counter()
    formatted_time = str(timedelta(seconds=int(END_TIME - START_TIME)))
    history['training_duration'].append(formatted_time)

    print(f"{'—'*75}")
    print(f"(✓) {config['epochs']} Epochs completed in {history['training_duration']} "
          f"at {time.strftime('%H:%M:%S', time.localtime())} with best Val F1: {best_f1:.4f}")

    return model, history, tracker

In [ ]:
def watch_parameter_change(model):
    """
    Snapshot weights before training. Compare after epoch 1.
    If weights haven't changed → that layer is not training.
    """
    # Call BEFORE training starts
    snapshots = {}
    base = model._orig_mod if hasattr(model, '_orig_mod') else model
    for name, param in base.named_parameters():
        snapshots[name] = param.data.clone()
    return snapshots

def compare_parameter_change(model, snapshots):
    """Call AFTER epoch 1 to see what changed."""
    print(f"\n{'='*60}")
    print("PARAMETER CHANGE CHECK (after epoch 1)")
    print(f"{'Layer':<40} | {'Changed?':<10} | {'Delta Norm'}")
    print(f"{'='*60}")
    
    base = model._orig_mod if hasattr(model, '_orig_mod') else model
    for name, param in base.named_parameters():
        if name in snapshots:
            delta = (param.data - snapshots[name]).norm().item()
            changed = delta > 1e-8
            status = "✅ Learning" if changed else "❌ FROZEN"
            print(f"  {name:<38} | {str(changed):<10} | {delta:.8f}  {status}")
    print(f"{'='*60}\n")

# HOW TO USE:
snapshots = watch_parameter_change(model)
# ... run 1 epoch ...
compare_parameter_change(model, snapshots)